In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# FFT 기초 — 수식 없이 주파수 분석 시작하기

진동 신호를 시간 영역에서 관찰하는 것만으로는 어떤 주기 성분이 들어 있는지 알기 어렵다.
이 노트북에서는 다음 세 단계로 주파수 분석의 기초 감각을 익힌다.

1. **실습 1.** 시간 영역에서 신호를 로드하고 일부 구간을 확대하여 주기성을 눈으로 확인한다.
2. **실습 2.** `np.fft.rfft` 를 직접 호출하여 단측 스펙트럼을 물리적 의미가 있는 진폭으로 스케일링한다.
3. **실습 3.** 교재에서 제공하는 `utils.fft` 로 결과를 재확인하고, 50~70 Hz 구간을 확대하여 bin 해상도와 누설(leakage) 효과를 체감한다.

수식은 다음 노트북부터 도입하며, 여기서는 먼저 "어떻게 보이는가"에 집중한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import utils

---

## 실습 1. 시간 영역 신호 로드 및 확대 관찰

샘플링 주파수 `fs = 1000 Hz` 로 1초 동안 취득된 진동 신호를 로드한다.
전체 구간(0~1 s)에서 한 번 그린 뒤, 0~0.2 s 구간으로 확대하여 반복되는 파형을 확인한다.

In [ ]:
fs = 1000

data = np.array(pd.read_csv('./data/data_sample_fft.csv'))
print('data shape:', np.shape(data))

t = data[:, 1]
v = data[:, 2]

fig, ax = plt.subplots(2, 1, figsize=(10, 5))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Full record (0 ~ 1 s)')

ax[1].plot(t, v, 'C0')
ax[1].set_xlim([0, 0.2])
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('Zoom-in (0 ~ 0.2 s)')

fig.tight_layout(); plt.show()

**관찰.** 전체 1초 구간에서는 값이 빠르게 오르내려 잡음처럼 보이지만,
0~0.2 s 구간으로 확대하면 약 60 Hz (주기 ≈ 1/60 s ≈ 16.7 ms) 의 주기 성분이 뚜렷이 관찰된다.
즉 신호는 "무질서"한 것이 아니라, 짧은 창으로 보면 구조가 드러난다.
이것이 주파수 분석이 필요한 첫 번째 이유다 — 시간 영역 관찰만으로는 성분의 개수·진폭을 정량화하기 어렵다.

---

## 실습 2. `rfft` 직접 구현 — 단측 스펙트럼 스케일링

실수 입력 신호의 FFT 는 대칭이므로 양의 주파수(단측, one-sided) 만 사용한다.
`numpy.fft.rfft` 는 길이 `N` 입력에 대해 `N//2 + 1` 개의 복소 계수를 반환한다.
원 신호의 진폭 단위를 복원하려면 다음 스케일링이 필요하다.

- 기본 스케일: `A = 2 * |X| / N`
- DC 성분 `A[0]` 은 두 번 세지지 않도록 2 로 나눈다.
- 짝수 `N` 의 Nyquist 성분 `A[-1]` 도 마찬가지로 2 로 나눈다.

시간 영역 신호와 단측 스펙트럼을 나란히 그려 60 Hz 피크의 진폭이 시간 영역 진폭과 일치하는지 확인한다.

In [ ]:
# rfft 로 단측 스펙트럼 계산 및 물리 진폭 스케일링
N = len(v)
X = np.fft.rfft(v)
f = np.fft.rfftfreq(N, d=1.0/fs)
A_manual = 2.0 * np.abs(X) / N
A_manual[0] /= 2.0                 # DC 반분
if N % 2 == 0:
    A_manual[-1] /= 2.0            # Nyquist 반분 (짝수 N)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Time Domain')

ax[1].plot(f, A_manual, 'C1')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('One-sided Spectrum (manual rfft)')

fig.tight_layout(); plt.show()

print('peak frequency  :', f[np.argmax(A_manual)], 'Hz')
print('peak amplitude  :', A_manual.max())

**관찰.** `rfft` 결과에 `2|X|/N` 스케일링을 적용하면 60 Hz 피크의 진폭이 시간 영역 파형의 진폭과 일치한다.
DC(`A[0]`) 와 Nyquist(`A[-1]`) 만 `2` 로 나누는 이유는, 이 두 bin 은 단측 스펙트럼에서 대응되는 음의 주파수 쌍이 없어 두 번 세지 않아야 하기 때문이다.
이 한 번의 스케일링 규칙만으로 "FFT 결과를 물리적 진폭으로 읽을 수 있게" 된다.

---

## 실습 3. `utils.fft` 활용 — 전체 스펙트럼 + 50~70 Hz 확대

실습 2 의 스케일링 절차는 교재 실습 전용 유틸 `utils.fft(v, fs)` 에 캡슐화되어 있다.
동일한 결과가 나오는지 `np.allclose` 로 수치 검증한 뒤,
전체 스펙트럼(plot) 과 60 Hz 주변(50~70 Hz) 확대(stem) 를 함께 그려 bin 간격 `fs/N` 과 이웃 bin 으로의 누설을 관찰한다.

In [ ]:
# utils.fft 로 동일 계산 + 수치 검증
f, A_util = utils.fft(v, fs)
print('bin spacing fs/N = ', fs / N, 'Hz')
print('np.allclose(A_manual, A_util) =', np.allclose(A_manual, A_util))

fig, ax = plt.subplots(3, 1, figsize=(10, 9))

ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('Amplitude')
ax[0].set_title('Time Domain')

# 포인트 수가 많을 때는 plot 이 가독성이 좋다
ax[1].plot(f, A_util, 'C1')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('Amplitude')
ax[1].set_title('Frequency Domain (full)')

# 확대 구간은 개별 bin 이 보이도록 stem 이 적합
ax[2].stem(f, A_util, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[2].set_xlim([50, 70])
ax[2].set_xlabel('Frequency (Hz)'); ax[2].set_ylabel('Amplitude')
ax[2].set_title('Frequency Domain (zoom 50 ~ 70 Hz)')

fig.tight_layout(); plt.show()

**관찰.** `np.allclose(A_manual, A_util)` 이 `True` 이므로 수작업 스케일링과 유틸 함수는 완전히 동일한 결과를 반환한다.
전체 스펙트럼에서는 60 Hz 한 지점만 눈에 띄지만, 50~70 Hz 로 확대하면 bin 간격이 `fs/N = 1000/1000 = 1 Hz` 임을 확인할 수 있고,
60 Hz 옆 bin(59, 61 Hz) 에도 작은 진폭이 남는 누설(spectral leakage) 을 볼 수 있다.
이 누설은 유한 관측 구간의 불가피한 효과로, 다음 노트북들에서 윈도우·필터링으로 완화하는 방법을 다룬다.

---